# Human-in-the-loop demo

**You are the robot.** The laptop's webcam is the robot's camera. Each time you run the step cell, the brain captures one frame, decides what the robot should do, and shows you a huge, color-coded Action. Then you physically move the laptop the way the action says — turn it left, walk it forward, etc. — and run the cell again.

If the brain works, you'll be guided directly to the target bottle in a handful of steps. If it doesn't work, you'll see exactly which step it broke (saturated detections, wrong direction, never stopping, etc.).

**Suggested setup:**
1. Put the target bottle on a table or floor with a few landmarks around it (chair, bench, door).
2. Stand 3-5 m away with the laptop, bottle out of camera view.
3. Run the setup cell once.
4. Run the step cell, follow the Action, repeat.

## Setup (run once)

In [ ]:
%matplotlib inline
import sys, time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle

REPO = Path.cwd()
if not (REPO / 'brain').exists() and (REPO.parent / 'brain').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

assert torch.cuda.is_available(), 'CUDA not available — fix torch install first'
print(f'GPU: {torch.cuda.get_device_name(0)}  ·  '
      f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM')

from brain.perception.target_finder import TargetFinder
from brain.perception.vlm_scout import VLMScout
from brain.control.loop import Action, ApproachController
from brain.control.action_to_pwm import pwm_for

REF = REPO / 'references' / 'ref.jpg'
CTX = REPO / 'references' / 'ctx.jpg'
assert REF.exists() and CTX.exists(), 'missing reference images — run tools/capture_references.py'

print('loading OWLv2...')
finder = TargetFinder(model_name='google/owlv2-base-patch16-ensemble', min_sim=0.3)
finder.load_reference(cv2.imread(str(REF)))

print('loading Qwen3-VL-8B (4-bit, 30-90s cold start)...')
t0 = time.monotonic()
scout = VLMScout(load_in_4bit=True)
print(f'  loaded in {time.monotonic() - t0:.1f}s  ·  '
      f'VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

controller = ApproachController(
    target_finder=finder,
    vlm_scout=scout,
    reference_photo=str(REF),
    reporter_photo=str(CTX),
)
history = []
print('\nready. Run the step cell below to take a frame and see the Action.')

## Tunables

Adjust before re-running the step cell. `min_sim` is the OWLv2 threshold; lower = more permissive. `WEBCAM_INDEX` is which camera to use (try 1 or 2 if 0 isn't your USB cam).

In [ ]:
WEBCAM_INDEX = 0
MIN_SIM = 0.3   # raise this if OWLv2 is detecting wall everywhere

finder.min_sim = MIN_SIM

## Step (run repeatedly)

Each run: capture one frame, decide one Action, show it. Then physically move the laptop the way the Action says, and run the cell again.

In [ ]:
ACTION_LABEL = {
    Action.FORWARD:      'WALK FORWARD',
    Action.LEFT:         'TURN LEFT',
    Action.RIGHT:        'TURN RIGHT',
    Action.STOP:         'STOP — TARGET REACHED',
    Action.SEARCH_LEFT:  'ROTATE LEFT (scanning for target)',
    Action.SEARCH_RIGHT: 'ROTATE RIGHT (scanning for target)',
}
ACTION_COLOR = {
    Action.FORWARD:      '#00cc00',  # green
    Action.LEFT:         '#ff8800',  # orange
    Action.RIGHT:        '#ff8800',
    Action.STOP:         '#cc0000',  # red
    Action.SEARCH_LEFT:  '#cc00cc',  # magenta
    Action.SEARCH_RIGHT: '#cc00cc',
}
ACTION_ARROW = {
    Action.FORWARD:      '\u2191',  # ↑
    Action.LEFT:         '\u2190',  # ←
    Action.RIGHT:        '\u2192',  # →
    Action.STOP:         '\u25A0',  # ■
    Action.SEARCH_LEFT:  '\u21BA',  # ↺
    Action.SEARCH_RIGHT: '\u21BB',  # ↻
}

def take_step():
    cap = cv2.VideoCapture(WEBCAM_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    for _ in range(5): cap.read()  # let autoexposure settle
    ok, frame = cap.read()
    cap.release()
    assert ok, 'webcam capture failed'

    t0 = time.monotonic()
    detections = finder.detect(frame)
    action = controller.step(frame)
    elapsed_ms = (time.monotonic() - t0) * 1000
    left, right = pwm_for(action)
    history.append((action, len(detections), elapsed_ms))

    fig = plt.figure(figsize=(14, 11))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1.2], hspace=0.05)

    ax_img = fig.add_subplot(gs[0])
    ax_img.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    if detections:
        d = detections[0]
        x1, y1, x2, y2 = d.xyxy
        ax_img.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                                   fill=False, edgecolor='lime', linewidth=4))
        ax_img.text(x1, max(y1-8, 16), f'target  {d.confidence:.2f}',
                    color='black', fontsize=14, weight='bold',
                    bbox=dict(facecolor='lime', alpha=0.85, pad=4))
    ax_img.axis('off')

    ax_act = fig.add_subplot(gs[1])
    ax_act.set_xlim(0, 1); ax_act.set_ylim(0, 1); ax_act.axis('off')
    color = ACTION_COLOR[action]
    ax_act.text(0.5, 0.65, f'{ACTION_ARROW[action]}  {ACTION_LABEL[action]}  {ACTION_ARROW[action]}',
                ha='center', va='center', fontsize=44, color=color, weight='bold')
    sub = (f'step #{len(history)}   ·   '
           f'{len(detections)} detection(s)   ·   '
           f'pwm = ({left:+d}, {right:+d})   ·   '
           f'{elapsed_ms:.0f} ms')
    ax_act.text(0.5, 0.2, sub, ha='center', va='center', fontsize=13, color='#666')
    plt.show()
    return action

_ = take_step()

**Re-run the cell above** every time you want a new decision. Move the laptop between runs.

What you should see (assuming the brain works):
- Bottle hidden → **ROTATE LEFT/RIGHT** (magenta) — the VLM has decided which way the bottle probably is. Rotate the laptop in that direction.
- Bottle now visible, off to the side → **TURN LEFT/TURN RIGHT** (orange). Turn the laptop slightly.
- Bottle visible and roughly centered → **WALK FORWARD** (green). Walk toward it.
- Bottle filling >15% of the frame → **STOP** (red). You're close enough — the robot would now drive over the bottle with its scoop.

If you reach STOP in <10 steps from anywhere in the room, the brain is working end-to-end.

## Trajectory log

What actions you've gotten in this session, in order. Useful for spotting patterns (e.g., the controller flipping back and forth between LEFT and RIGHT = it's oscillating).

In [ ]:
if not history:
    print('no steps taken yet')
else:
    print(f'{"#":>3}  {"action":<14}  {"dets":>4}  {"step ms":>8}')
    print('-' * 38)
    for i, (act, n, ms) in enumerate(history, 1):
        print(f'{i:>3}  {act.name:<14}  {n:>4}  {ms:>8.0f}')

    # quick pattern check
    last5 = [a.name for a, _, _ in history[-5:]]
    if len(last5) == 5 and len(set(last5)) > 3:
        print('\n⚠️  last 5 actions are noisy — controller may be oscillating.')
    if last5 == ['STOP'] * 5:
        print('\n✅ stable STOP — would now begin INTAKING (drive forward into bottle).')

## Reset (start a fresh run)

In [ ]:
history.clear()
# reset the controller's internal search-burst counter so a fresh run starts clean
controller._search_remaining = 0
controller._search_direction = None
print('history cleared, controller reset. Position yourself and run the step cell.')